# Build Dataset v3 — CS 231N + 3000 COCO Negatives
Añade 3000 imágenes negativas de COCO al dataset de detección de armas.  
- **Input dataset**: `weapons_cs231n/raw_zip/CS 231N Project.v2i.yolov8.zip`
- **Input COCO**: `coco_raw/train2017/` (ya extraído en Drive)
- **Output**: `weapons_cs231n/raw_zip/CS 231N Project.v3i.yolov8.zip`

## 1. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Config

In [ ]:
import os

BASE         = '/content/drive/MyDrive/TFM/datasets'

ORIGINAL_ZIP = f'{BASE}/weapons_cs231n/raw_zip/CS 231N Project.v2i.yolov8.zip'
COCO_DIR     = f'{BASE}/coco_raw/train2017'   # carpeta ya extraída, no zip
OUTPUT_ZIP   = f'{BASE}/weapons_cs231n/raw_zip/CS 231N Project.v3i.yolov8.zip'

N_NEGATIVES  = 3000
TARGET_SIZE  = (640, 640)  # mismo preproceso que el dataset original
SEED         = 42

print('Original zip exists:', os.path.exists(ORIGINAL_ZIP))
print('COCO dir exists:    ', os.path.isdir(COCO_DIR))
print('COCO images found:  ', len([f for f in os.listdir(COCO_DIR) if f.endswith('.jpg')]))

## 3. Samplear imágenes COCO

In [ ]:
import random

random.seed(SEED)

all_coco = [f for f in os.listdir(COCO_DIR) if f.endswith('.jpg')]
sampled  = random.sample(all_coco, N_NEGATIVES)

print(f'Total disponibles: {len(all_coco)}')
print(f'Seleccionadas:     {len(sampled)}')

## 4. Construir el ZIP de salida

In [ ]:
import zipfile
import io
from PIL import Image

print('Construyendo ZIP de salida...')

with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as out:

    # --- Copiar dataset original completo ---
    with zipfile.ZipFile(ORIGINAL_ZIP) as orig:
        names = orig.namelist()
        for i, item in enumerate(names):
            if i % 2000 == 0:
                print(f'  Copiando original [{i}/{len(names)}]...')
            out.writestr(item, orig.read(item))

    # --- Añadir negativos de COCO (leídos desde carpeta extraída) ---
    print(f'  Añadiendo {N_NEGATIVES} negativos COCO...')
    for i, fname in enumerate(sampled):
        if i % 300 == 0:
            print(f'    [{i}/{N_NEGATIVES}] procesando...')

        img_path = os.path.join(COCO_DIR, fname)
        img = Image.open(img_path).convert('RGB')
        img = img.resize(TARGET_SIZE, Image.LANCZOS)

        buf = io.BytesIO()
        img.save(buf, format='JPEG', quality=85)
        buf.seek(0)

        base = fname.replace('.jpg', '')
        out.writestr(f'train/images/coco_neg_{base}.jpg', buf.getvalue())
        out.writestr(f'train/labels/coco_neg_{base}.txt', '')  # vacío = negativo

print(f'\n✓ ZIP generado: {OUTPUT_ZIP}')

## 5. Verificación

In [ ]:
print('Verificando ZIP de salida...')
with zipfile.ZipFile(OUTPUT_ZIP) as z:
    names     = z.namelist()
    train_img = sum(1 for n in names if n.startswith('train/images/') and n.endswith('.jpg'))
    train_lbl = sum(1 for n in names if n.startswith('train/labels/') and n.endswith('.txt'))
    neg_img   = sum(1 for n in names if 'coco_neg_' in n and n.endswith('.jpg'))

print(f'  train/images total : {train_img}')
print(f'  train/labels total : {train_lbl}')
print(f'  negativos añadidos : {neg_img}')
print(f'  % negativos        : {neg_img / train_img * 100:.1f}%')

size_mb = os.path.getsize(OUTPUT_ZIP) / 1024**2
print(f'  Tamaño ZIP         : {size_mb:.0f} MB')